# S&P 500 Options: Holdout Predictions

**Chapter 20 - Out-of-sample evaluation**

Every number in this case study so far was measured on the validation folds, and every
choice was made by looking at them: which model family, which configuration, how many
straddles to write each week, how to size them, what to charge for the option spread and
the hedge. A result selected that way cannot also be evidence that the selection was
sound - the ranking and the evidence would be the same measurement.

The holdout is the window nothing has been selected on. This notebook fits the selected
configuration on the history available before that window opens and writes its predictions
over it. [`17_holdout_backtest`](17_holdout_backtest.ipynb) turns those predictions into
short straddles and a return series, with the concentration, the allocator and the cost
assumption the case study settled on, and
[`18_strategy_analysis`](18_strategy_analysis.ipynb) reads both back.

**What this notebook is careful about**

A holdout prediction is not the validation model scored on a later window. Section 2 fits
again, over a training interval that ends before the window opens, and the new training
identity is what makes the refit visible rather than asserted: the identity covers the CV
interval, so a run that came back with the validation training hash would mean no refit
happened. The check is in section 3 and it raises.

The label makes the gap between training and the window unusually long here. `ret_to_expiry`
resolves when a contract expires, not a fixed number of sessions after the decision, so the
buffer is declared in `setup.yaml` as `labels.buffer` and is a full option cycle rather than
a formality.

**Prerequisites:** [`15_costs`](15_costs.ipynb), which is the last stage that selects.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# S&P 500 選擇權：保留測試集預測

**第 20 章：樣本外評估**

本案例研究至今的每個數字都來自驗證摺疊，而每項選擇也都是看過這些數字後決定：模型家族、組態、每週賣出多少組跨式組合、如何決定部位規模、選擇權價差及避險要計入多少成本。依此選出的結果，不能再用來證明選取本身可靠，否則排序依據與證據就是同一份量測。

保留測試集是未曾用來選取任何項目的時間窗。本 Notebook 以時間窗開始前可用的歷史資料重新擬合選定組態，並產生該時間窗內的預測。[`17_holdout_backtest`](17_holdout_backtest.ipynb) 依案例研究確定的持倉集中度、配置方法與成本假設，將這些預測轉為空頭跨式組合及報酬序列；[`18_strategy_analysis`](18_strategy_analysis.ipynb) 則回讀兩者。

**本 Notebook 特別注意的事項**

保留測試集預測不是將驗證模型拿到較晚的時間窗評分。第 2 節會在時間窗開始前結束的訓練區間重新擬合；新的訓練身分識別值使重新擬合可被檢查，而非只靠宣稱：該識別值涵蓋交叉驗證區間，因此若執行結果仍使用驗證訓練雜湊值，就表示沒有重新擬合。第 3 節會檢查並在異常時引發錯誤。

此處的標籤使訓練與時間窗之間的間隔特別長。`ret_to_expiry` 在合約到期時才確定，而非決策後固定幾個交易日，因此緩衝區在 `setup.yaml` 宣告為 `labels.buffer`，長達一個完整選擇權週期，並非形式上的設定。

**先備條件：** [`15_costs`](15_costs.ipynb)，亦即最後一個進行選取的階段。
<!-- ML4T_ZH_END c0000-p000 -->

**Scope:** one training run and one prediction set. No backtest, no straddles, no
selection, no comparison - those are 17 and 18.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**範圍：**一次訓練執行及一組預測集。回測、跨式組合、選取及比較均由 17 與 18 處理。
<!-- ML4T_ZH_END c0000-p001 -->


In [ ]:
"""S&P 500 Options: Holdout Predictions."""

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.registry.maintenance import delete_prediction_generation
from case_studies.utils.strategy_analysis import (
    holdout_generations_to_retire,
    registered_holdout_generations,
    resolve_solvent_carrier,
)
from utils.paths import get_case_study_dir

In [ ]:
CASE_STUDY_ID = "sp500_options"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
# Replace a holdout evaluation this window already carries, rather than adding a second one.
# Off by default, because the default should be the methodology: one configuration is measured
# on the holdout, and the number means what it says only while that stays true. It is a
# parameter and not a prohibition because a holdout does get re-derived - a corrected input
# upstream moves every training identity, and the evaluation that was right for the old
# geometry is then an evaluation of a configuration this case study no longer selects.
# Refusing that outright would mean a correction could never be carried through to the end.
REPLACE_HOLDOUT = False

In [ ]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)


def _delete_holdout_generation(case_dir, prediction_hash):
    """Remove one holdout prediction set and everything registered against it.

    Reached only when ``REPLACE_HOLDOUT`` says a generation is superseded. The rows go rather
    than being marked, because a superseded holdout evaluation that is still readable is still
    a number someone can quote, and the point of replacing it is that it should not be one.

    The cascade lives in `case_studies/utils/registry/maintenance.py` and derives the child
    tables from the schema, because the version of this that listed them by hand missed
    `cohort_metrics.leader_hash` - which with foreign keys enabled aborts the delete rather
    than orphaning a row.
    """
    deleted = delete_prediction_generation(case_dir / "run_log" / "registry.db", prediction_hash)
    for table, n in sorted(deleted.items()):
        print(f"  deleted {n:>3} from {table}")

## 1. Which configuration the holdout runs

The holdout runs the configuration the case study reports, resolved through the same
`resolve_solvent_carrier` [`15_costs`](15_costs.ipynb) prices. Resolving it again here
rather than passing it along is deliberate: the two notebooks must agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the cross-stage
validation rank-1, and it was fixed before this notebook ran.

**Its validation Sharpe is negative.** That is this case study's result, not a reason to look for
a different configuration: writing straddles on a signal this weak does not pay for the option
spread and the hedge, and the holdout is being used to see whether that reading holds on a window
nothing was chosen on. Picking a configuration for its sign would be the selection the holdout
exists to make honest.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留測試集使用哪個組態

保留測試集使用案例研究報告的組態，並透過相同的 `resolve_solvent_carrier` 取得；[`15_costs`](15_costs.ipynb) 也是透過它定價。在此重新解析而非直接傳遞，是刻意的安排：兩份 Notebook 必須依設計取得一致結果；若一份寫下雜湊值、另一份讀取，只有在掃描尚未重新建立時才會一致。

保留測試集的任何資訊都不參與選取。選定組態是跨階段驗證排名第一的組態，在本 Notebook 執行前就已固定。

**其驗證夏普比率為負。** 這是本案例研究的結果，並非尋找其他組態的理由：以如此微弱的訊號賣出跨式組合，收益不足以支付選擇權價差及避險成本；保留測試集的用途，是檢查這項解讀在未曾參與選取的時間窗是否仍成立。若因夏普比率的正負而改選組態，正是保留測試集要避免的不誠實選取。
<!-- ML4T_ZH_END c0004-p000 -->


In [ ]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

The checkpoint is part of the configuration. Where a family publishes a prediction set per
checkpoint on a declared schedule, the selected configuration's prediction set names one of them,
and refitting without it would produce a model at the end of training rather than the one that was
ranked. A family with no checkpoint dimension - which is where this case study's configuration
sits - stores NULL in both columns and carries that NULL through unchanged.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是組態的一部分。若某家族按預先宣告的時程，為每個檢查點發布一組預測集，選定組態的預測集會指定其中一個；若重新擬合時忽略它，產生的就會是訓練結束時的模型，而非當初排名的模型。沒有檢查點維度的家族，也就是本案例選定組態所在的家族，會在兩個欄位都儲存 NULL，並原樣傳遞該值。
<!-- ML4T_ZH_END c0006-p000 -->


In [ ]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from this case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the three
cannot disagree.

The training interval is everything available before that window, bounded above by the label
buffer. **For this case study the buffer is doing real work.** `ret_to_expiry` is realised
at expiration rather than a fixed horizon after the decision, so a row dated `t` records an
outcome that is not known until its contract expires - and a training set running to the day
the window opens would be fitted on labels that resolve inside it. The buffer is therefore an
option cycle's worth, and the derivation refuses to default it: a zero gap here would be a
leak.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a fold's
own training rows - are recomputed against the holdout fold. Carrying those forward would fit
a model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 時間窗與允許看見它的模型

保留測試集時間窗不是在此選定的。其範圍為 `evaluation.holdout_start` 與 `evaluation.holdout_end`，取自本案例自身的 `setup.yaml`；讀取時使用與摺疊推導及回測切片相同的 `canonical_window`，使三者不可能不一致。

訓練區間包含時間窗之前所有可用資料，但上界受標籤緩衝區限制。**在本案例中，緩衝區確實發揮作用。** `ret_to_expiry` 在到期時才實現，而非決策後固定的時間長度，因此日期為 `t` 的資料列，記錄的是合約到期前仍未知的結果；若訓練集延伸到時間窗開始當天，就會使用在該時間窗內才確定的標籤進行擬合。緩衝區因此長達一個選擇權週期，推導程序也拒絕使用預設值：此處若間隔為零，就會造成資訊洩漏。

組態的其他部分均原樣傳遞；無法直接傳遞的欄位，包括合格資格清單，以及此家族依各摺疊自身訓練資料列決定的參數，都會針對保留測試集摺疊重新計算。若沿用舊值，就會擬合一個以驗證摺疊為依據的模型，卻稱其為重新訓練。
<!-- ML4T_ZH_END c0008-p000 -->


In [ ]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains  {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends
# before the holdout opens. Printing both is what lets a reader check the gap rather than take
# it on the derivation's word.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")

## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the spec above. Its name comes
from a locked holdout path this case study does not use; it takes a training specification
and a checkpoint, not a lock, and it is used here because it is the one call that refuses a
request that is not exactly the spec it was handed - the training identity, the checkpoint
schedule, the feature lineage and the runtime parameters are all checked before anything is
fitted.

The training identity below is new. It has to be: it covers the CV interval, and the holdout
fold is not one of the validation folds. A run that came back with the validation training
hash would mean the refit did not happen, so that is checked rather than assumed. **That is
not a hypothetical here.** Two holdout prediction sets were registered against this case
study under the validation training identity, by a driver that refits the model and then
registers the result under the identity it started from; their split said `holdout` and
their model had been fitted on folds ending inside the window. They were removed, and the
check below is what makes their shape impossible to reintroduce.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 3. 擬合並登錄預測

`reconstruct_locked_model_request` 依上述規格建立請求。其名稱來自本案例未使用的鎖定保留測試集路徑；它接收訓練規格與檢查點，而非鎖定檔。此處使用它，是因為唯有這項呼叫會拒絕與所給規格不完全一致的請求：在任何擬合開始前，它會檢查訓練身分識別值、檢查點時程、特徵來源鏈及執行期參數。

下方訓練身分識別值是新的，而且必須如此：它涵蓋交叉驗證區間，而保留測試集摺疊不屬於驗證摺疊。若執行回傳驗證訓練雜湊值，就表示未重新擬合，因此此處會實際檢查。**這在本案例不是假設性的風險。** 曾有兩組保留測試集預測以驗證訓練身分識別值登錄到本案例：某個驅動程式重新擬合模型後，仍以原先的識別值登錄結果；其資料切分標示為 `holdout`，而模型曾以延伸至時間窗內的摺疊擬合。這兩組結果已移除；下方檢查可防止同類結果再次出現。
<!-- ML4T_ZH_END c0010-p000 -->

**The window carries one configuration at a time.** The check below is on the selected
configuration rather than on the notebook, and it has three outcomes. With the selected
configuration unchanged this is an idempotent replay: the derivation is deterministic and the
training identity covers it, so the same identity comes back and the fit is served from the
registry, which is why re-running the notebook is free and safe. With the selected configuration
changed it stops and names both configurations. With `REPLACE_HOLDOUT` set it replaces the
earlier generation instead of standing beside it, so the registry never holds two refits of the
same window.

What the replacement does not do is undo having observed the earlier result, and that is the
part worth understanding rather than enforcing. A holdout number is out-of-sample because
nothing about the model was chosen after seeing it. Evaluate a second configuration on the
same window and the second number is no longer that, however the registry is arranged: the
selection may have been informed by the first. Deleting rows removes the evidence, not the
knowledge.

So the honest use of this switch is narrow: an input upstream was corrected, every training
identity moved with it, and the evaluation being replaced belongs to a configuration this
case study no longer selects. That is maintenance, and it has to be possible or a correction
could never reach the end of the pipeline. Reaching for it because the first holdout
disappointed is the thing that makes an out-of-sample claim false, and no parameter default
can tell those two apart - the researcher can, and it is their call.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
**每個時間窗一次只容納一種組態。** 下方檢查針對選定組態，而非 Notebook，有三種結果。若選定組態不變，這是具冪等性的重播：推導過程具決定性，訓練身分識別值涵蓋其內容，因此會取得相同識別值，擬合結果也由登錄系統提供；這就是重新執行 Notebook 不額外耗費訓練、且安全的原因。若選定組態改變，流程會停止並列出兩種組態。若設定 `REPLACE_HOLDOUT`，則會取代前一代結果，而非並列，使登錄系統不會同時持有同一時間窗的兩次重新擬合。

替換並不能抹去研究者已經看過先前結果這件事；這正是需要理解而非只靠機制強制的部分。保留測試集結果之所以是樣本外，是因為模型的任何部分都未在看到結果後才選定。若在同一時間窗評估第二種組態，無論登錄系統如何安排，第二個數字都不再符合這項條件：選取可能受第一個數字影響。刪除資料列只能移除證據，無法抹除已知資訊。

因此，誠實使用此開關的情況很有限：上游輸入經修正、所有訓練身分識別值都隨之變更，而被替換的評估屬於本案例已不再選取的組態。這是維護工作，而且必須允許，否則修正永遠無法傳遞到流程末端。若因第一次保留測試集結果不理想而使用它，樣本外宣稱就會失真；任何參數預設值都無法區分這兩種情況，只有研究者能判斷，並由研究者負責。
<!-- ML4T_ZH_END c0010-p001 -->


In [ ]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
retire = holdout_generations_to_retire(CASE_DIR, this_generation=this_generation)
# A row whose training run records no CV split cannot be shown either way, and deleting on
# that would discard a result nothing has established is wrong. It stops the run instead.
if retire.unattributable:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs record no CV split, "
        "so whether they were refitted for the holdout cannot be established: "
        + ", ".join(
            f"{row['prediction_hash']} (training {row['training_hash']})"
            for row in retire.unattributable
        )
        + ". Establish what produced them before registering another evaluation on the same "
        "window; this notebook will not delete a row it cannot show is not a holdout result."
    )
# A row whose training run declares a non-holdout CV may not be reported as a holdout
# result, and it is also not something to delete unattended: `generate_holdout` refits on a
# holdout fold and then registers the predictions under the VALIDATION training identity, so
# this record covers both a validation-fitted model published over the window and a real
# refit filed under the wrong identity. Nothing owned this before - the filter here was
# `row["refitted"]`, which made exactly these rows invisible to the refusal and to
# everything after it.
if retire.not_out_of_sample and not REPLACE_HOLDOUT:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs declare a CV split "
        "other than the holdout: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in retire.not_out_of_sample
        )
        + ". Each is either a validation-fitted model published over the window, which is "
        "not an out-of-sample result, or a refit registered under its validation training "
        "identity, which the retired `20_strategy_synthesis/holdout.py::generate_holdout` "
        "wrote until it was deleted on 2026-09-12 - and "
        "the registry cannot tell those apart. Establish which, then set REPLACE_HOLDOUT="
        "True to remove it, or leave it and resolve the identity instead."
    )
for row in retire.not_out_of_sample:
    print(
        f"REMOVING {row['prediction_hash']} ({row['config_name']}, training "
        f"{row['training_hash']}): its training run declares a non-holdout CV, so it is not "
        "reportable as a holdout evaluation under the identity it carries"
    )
    _delete_holdout_generation(CASE_DIR, row["prediction_hash"])
superseded = list(retire.superseded)
if superseded and not REPLACE_HOLDOUT:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the "
        "same window, which is a second configuration measured on a period this case study "
        "reports as unseen. Set REPLACE_HOLDOUT=True to replace the earlier generation - "
        "correct when an upstream fix moved every training identity and the evaluation being "
        "replaced belongs to a configuration no longer selected - or leave the selection "
        "where it was."
    )
for row in superseded:
    print(f"REPLACING holdout generation {row['prediction_hash']} ({row['config_name']})")
    _delete_holdout_generation(CASE_DIR, row["prediction_hash"])

In [ ]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:  {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

What the prediction set covers, read back from the registry rather than from the request.
The two agree only if the fit published what it declared. The unit here is a contract on a
date, not a name on a date: one underlying carries many strikes and expirations at once, so
the row count is far larger than the session count and the symbol count is underlyings
rather than tradeable instruments.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
從登錄系統回讀預測集涵蓋的範圍，而非只讀請求。只有擬合程序確實發布其宣告的內容，兩者才會一致。此處的計數單位是某日期的一份合約，而非某日期的一個標的名稱：同一標的資產同時有多個履約價與到期日，因此資料列數遠大於交易日數，而標的代號數計的是標的資產，不是可交易工具。
<!-- ML4T_ZH_END c0013-p000 -->


In [ ]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(f"rows={predictions.height:,}  sessions={predictions['timestamp'].n_unique():,}")
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions['symbol'].n_unique():,} underlyings"
)

Every holdout prediction set the registry holds, and whether the model behind it was
refitted for the window. All of them are listed rather than one silently preferred, because
the registry is immutable and a reader looking at it later will see whatever is there. A row
marked VALIDATION-FITTED is not an out-of-sample result whatever its numbers say.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
列出登錄系統中的每組保留測試集預測，以及背後模型是否針對該時間窗重新擬合。此處列出全部結果，不會默默偏好其中一組，因為登錄系統不可變，日後讀取的人會看到當時存在的所有內容。標為 VALIDATION-FITTED 的資料列，無論數字如何，都不是樣本外結果。
<!-- ML4T_ZH_END c0015-p000 -->


In [ ]:
for row in registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends a full option cycle before
the window opens. That is a precondition for an out-of-sample claim, not the claim itself.
Nothing here says whether the predictions are any good - they have not been scored, turned
into straddles, or traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached this
notebook through a selection made on the validation folds, and this window is being used once
per configuration that gets here. What it does remove is the specific circularity of scoring
a validation-fitted model on the period meant to judge it - which, for this case study, is
not an abstract risk: it had already happened once.

Re-running this notebook is free: the same configuration re-derives the same training identity and
the fit is served from the registry. Evaluating a DIFFERENT configuration is not, and is
refused here. If a later pass finds the selection was wrong, that is a question for the
registry's lifecycle, which records that a second look was taken - not something to settle by
deleting rows until the registry agrees.

**Next:** [`17_holdout_backtest`](17_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 證明了什麼，以及尚未證明什麼

它證明一件事：本案例選定的組態，使用在時間窗開始前一個完整選擇權週期就已結束的資料擬合，並產生涵蓋保留測試集時間窗的預測集。這是樣本外宣稱的前提，並非宣稱本身。此處尚未判斷預測品質，因為預測尚未評分、轉為跨式組合或實際交易。

它也未使保留測試集成為嚴格意義下全新的測試。組態經由驗證摺疊上的選取才進入本 Notebook，而每個進入此處的組態會使用此時間窗一次。它確實消除了以驗證資料擬合的模型，在原本用來評判它的期間評分所造成的特定循環；對本案例而言，這並非抽象風險，因為先前確實發生過一次。

重新執行本 Notebook 不會重做擬合：同一組態會重新推導出相同訓練身分識別值，並由登錄系統提供擬合結果。但評估*不同*組態並非如此，因此此處會拒絕。若日後發現選取錯誤，應透過登錄系統記錄再次檢視的生命週期處理，而非持續刪除資料列直到登錄系統顯示想要的結果。

**下一步：** [`17_holdout_backtest`](17_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
